# Volumetric arena experiment

Test of the 3D network in a cubic arena. The walk is a reflecting random walk in the box, and the true plane stays horizontal.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt

# %cd /content
# !rm -rf /content/CAN_Path_Integration_3D_model
# !git clone -b two-dimensional --single-branch https://github.com/jacobsvennevik/CAN_Path_Integration_Plane_estimation_3D_model.git /content/CAN_Path_Integration_3D_model
# !pip install -q git+https://github.com/jacobsvennevik/MADE.git
# %cd /content/CAN_Path_Integration_3D_model



sys.path.insert(0, os.path.abspath("../.."))

from config import RunConfig, PlaneConfig, NetworkConfig, AnalysisConfig
from experiments.arena_3d import Arena3DExperiment, Arena3DConfig
from analysis.rate_maps import occupancy_fraction
from analysis.qy_fit import qy_fit
from analysis.structure3d import score_3d_from_map
from analysis.summaries import summarize_3d, summarize_qy
from model.network.visualize3D import plot_pi_error
from model.plane_estimation import held_normal_stats


## 2. Settings



In [ ]:
# --- arena and walk ---
N_STEPS      = 1_000_000
ENV_SIZE     = 2.0
GRID_SPACING = 0.48          # metres per field spacing; arena is ~4.17 periods
TARGET_SPEED = 0.004
SEEDS        = (0,)

# --- plane sweep (one run per pair) ---
KAPPA_SENS_VALUES  = (1000.0, 300.0, 30.0, 10.0)
TAU_REFRESH_VALUES = (100, 1000, 10000)
SHOW               = (SEEDS[0], 300.0, 100)
SIGMA              = AnalysisConfig.smooth_sigma
N_JOBS             = max(1, (os.cpu_count() or 1) - 1)

# --- network (3D sweep, trial 62) ---
net = NetworkConfig(
    spacing=0.1,
    dim=3,
    lambda_net=0.990,
    ratio=1.344,
    alpha=0.449,
    b=1.0,
    offset_magnitude=0.129,
    dt=0.25,
    tau=2.898,
    velocity_gain=1.401,
    settle_steps=2000,
)

# --- scoring ---
RATEMAP_BINS = 20
N_SUB        = 300       # cells scored; the full 3D sheet is too large for a 720-plane sweep
N_SHUFFLE    = 20        # circular shifts for spatial-information and sparsity z-scores
G_VEC        = np.array([0.0, 0.0, -9.81])

print("velocity_gain", net.velocity_gain, "lambda_net", net.lambda_net)
print("settle_steps", net.settle_steps)


## 3. Network

Build the experiment.


In [ ]:
plane = PlaneConfig(
    kappa_v=KAPPA_SENS_VALUES[0],
    kappa_sens=KAPPA_SENS_VALUES[0],
    plane_mode="bayesian",
    tau_refresh=TAU_REFRESH_VALUES[0], seed=0,
)
exp_cfg = Arena3DConfig(
    env_size=ENV_SIZE, n_steps=N_STEPS, seed=SEEDS[0],
    grid_spacing=GRID_SPACING, target_speed_rad_per_time=TARGET_SPEED,
    ratemap_bins=RATEMAP_BINS,
    ratemap_n_sub=N_SUB, ratemap_n_shuffle=N_SHUFFLE,
)
cfg = RunConfig(network=net, experiment=exp_cfg, plane=plane)
exp = Arena3DExperiment(cfg)


## 4. Trajectory

Generate the walk once per seed and reuse it for every (κ_sens, τ) pair. The figure shows coverage in three projections. Occupancy in the title is the fraction of voxels visited.


In [ ]:
def make_walk(seed):
    return exp.generate_trajectory(seed=int(seed), n_steps=N_STEPS)

traj0 = make_walk(SEEDS[0])
occ0 = occupancy_fraction(traj0.world_pos, ENV_SIZE, RATEMAP_BINS, ndim=3)
step = max(1, len(traj0.world_pos) // 20_000)
p = traj0.world_pos[::step]
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, (i, j, xi, yi) in zip(
        axes, ((0, 1, "x", "y"), (0, 2, "x", "z"), (1, 2, "y", "z"))):
    ax.plot(p[:, i], p[:, j], lw=0.3)
    ax.set_aspect("equal")
    ax.set_xlabel(xi)
    ax.set_ylabel(yi)
fig.suptitle(f"seed {SEEDS[0]}  occupancy {occ0:.1%}")
fig.tight_layout()
plt.show()


## 5. Runs

The same walk is used for every plane setting. Scores should stay fairly consistent across those settings.

Each run is scored with spatial information, the sparsity index, inter-field distance, the FCC / HCP / columnar structure scores, and the modified radial autocorrelation. `qy_fit` scores the same field trace. The walk is one route on the horizontal plane.


In [ ]:
def plane_for(kappa_sens, tau_refresh):
    return PlaneConfig(
        kappa_v=float(kappa_sens),
        kappa_sens=float(kappa_sens),
        plane_mode="bayesian",
        tau_refresh=int(tau_refresh),
        seed=0,
    )


def one_flight(traj):
    """Label the whole walk as one route on the horizontal plane."""
    n = len(traj.world_pos)
    traj.flight_of = np.zeros(n, dtype=int)
    traj.flight_table = [dict(flight=0, route=0, t0=0, t1=n)]
    traj.route_table = [dict(route=0, n_true=np.array([0.0, 0.0, 1.0]))]
    return traj


runs = {}
for seed in SEEDS:
    traj = one_flight(make_walk(seed))
    world_pos, v_body, torus_gt = traj
    n_true = getattr(traj, "n_true_seq", None)
    starts = getattr(traj, "segment_starts", None)

    for kappa_sens in KAPPA_SENS_VALUES:
        for tau_refresh in TAU_REFRESH_VALUES:
            print(f"\n=== seed {seed}  κ_sens {kappa_sens:g}  τ {tau_refresh} ===")
            res = exp.run(
                world_pos, v_body, torus_gt, G_VEC,
                n_true_seq=n_true, segment_starts=starts,
                plane=plane_for(kappa_sens, tau_refresh),
            )
            scores = score_3d_from_map(
                res.ratemap_sums, res.ratemap_counts,
                shuf_sums=res.shuf_sums,
                active_mask=res.active_mask,
                positions=res.field_pos,
                activity=res.field_act,
                grid_spacing=GRID_SPACING,
                env_size=ENV_SIZE,
                sigma=SIGMA,
                n_jobs=N_JOBS,
            )
            if n_true is None:
                n_true_arr = np.broadcast_to(
                    np.array([0.0, 0.0, 1.0]), (len(world_pos), 3))
            else:
                n_true_arr = np.asarray(n_true, dtype=float)
            stats = held_normal_stats(res.n_held_hist, n_true_arr, res.refresh_mask)
            lattice = qy_fit(
                traj, res.field_pos, res.field_act, res.field_stride,
                seed=int(seed), n_jobs=N_JOBS,
            )
            row = {
                **summarize_3d(scores, GRID_SPACING, ENV_SIZE),
                **summarize_qy(lattice),
            }
            print(
                f"median χ FCC {row['median_chi_fcc']:.3f}  "
                f"HCP {row['median_chi_hcp']:.3f}  "
                f"COL {row['median_chi_col']:.3f}  "
                f"grid fit {row['median_grid_fit']:.3f}  "
                f"significant {row['frac_significant']:.1%}  "
                f"final error {res.error_rad[-1]:.4f} rad  "
                f"occupancy {row['occupancy']:.1%}"
            )
            runs[(int(seed), float(kappa_sens), int(tau_refresh))] = dict(
                result=res, scores=scores, lattice=lattice, row=row, filter_stats=stats,
            )

out = runs[SHOW]
vol = np.asarray(out["result"].warmup_volume)
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, axis, lab in zip(axes, (2, 1, 0), ("sum z (xy)", "sum y (xz)", "sum x (yz)")):
    ax.imshow(vol.sum(axis=axis).T, origin="lower", cmap="inferno")
    ax.set_title(lab)
    ax.axis("off")
fig.suptitle(f"settled sheet  κ_sens {SHOW[1]:g}  τ {SHOW[2]}")
fig.tight_layout()
plt.show()


## 6. Path-integration accuracy

Final path-integration error against κ_sens, one line per refresh interval. The error is the wrapped distance on the torus, in radians.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for tau in TAU_REFRESH_VALUES:
    ys = [runs[(SEEDS[0], float(k), int(tau))]["result"].error_rad[-1]
          for k in KAPPA_SENS_VALUES]
    ax.plot(KAPPA_SENS_VALUES, ys, marker="o", label=f"τ={tau}")
ax.set_xscale("log")
ax.set_xlabel("κ_sens")
ax.set_ylabel("final error (radians)")
ax.set_title(f"seed {SEEDS[0]}  path-integration error")
ax.legend()
fig.tight_layout()
plt.show()


## 7. Path integration

Decoded torus position against ground truth for the run at κ_sens = 300, τ = 100. The printed RMS is the held-normal error, not the κ you set. The true normal stays horizontal for the whole walk. Gravity folds start below κ ≈ 10.


In [ ]:
res = out["result"]
fig, axes = plot_pi_error(res.torus_gt, res.theta_hist)
plt.show()
stats = out["filter_stats"]
print(f"κ_sens {SHOW[1]:g}  τ {SHOW[2]}  "
      f"rms={stats['rms_err_deg']:.2f} deg  "
      f"mean_rate={stats['mean_rate_deg']:.3f} deg/step  "
      f"refreshes={stats['refresh_count']}")
print(f"n_gravity_flips={res.n_gravity_flips}")


## 8. Scoring

One row per (κ_sens, τ_refresh). Scoring includes IFD, structure scores, spatial information, sparsity z-scores, and the modified radial autocorrelation. The Qi and Yartsev columns are the median grid-fit score and the fraction of cells the fit calls significant.


In [ ]:
rows = []
for (seed, kappa_sens, tau_refresh), run in runs.items():
    res = run["result"]
    rows.append(dict(
        seed=seed, kappa_sens=kappa_sens, tau_refresh=tau_refresh,
        final_error_rad=float(res.error_rad[-1]),
        tracking_score=float(res.tracking_score),
        rms_err_deg=run["filter_stats"]["rms_err_deg"],
        **run["row"],
    ))
display(pd.DataFrame(rows))


## 9. Figures

Four figures for κ_sens = 300, τ = 100. Structure scores, spatial information and sparsity, pooled inter-field distance with a line at one grid spacing, and the median modified radial autocorrelation.


In [ ]:
sc = out["scores"]
chi = np.asarray(sc["chi"], dtype=float)
fig, ax = plt.subplots(figsize=(6, 3.5))
for row, name in zip(chi, ("FCC", "HCP", "COL")):
    vals = row[np.isfinite(row)]
    if vals.size:
        ax.hist(vals, bins=20, alpha=0.55, label=name)
ax.set_xlabel("structure score")
ax.set_title(f"κ_sens {SHOW[1]:g}  τ {SHOW[2]}  per-cell structure scores")
ax.legend()
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, key, label in zip(axes, ("sinfo", "sidx"), ("spatial information", "sparsity index")):
    vals = np.asarray(sc[key], dtype=float)
    vals = vals[np.isfinite(vals)]
    ax.hist(vals, bins=20, color="#2166ac", alpha=0.8)
    ax.set_xlabel(label)
fig.suptitle(f"κ_sens {SHOW[1]:g}  τ {SHOW[2]}")
fig.tight_layout()
plt.show()

ifd = sc.get("ifd") or {}
pooled = [np.asarray(v, dtype=float) for v in ifd.values() if len(v)]
pooled = np.concatenate(pooled) if pooled else np.array([])
spacing_n = GRID_SPACING / (ENV_SIZE / 2.0)
fig, ax = plt.subplots(figsize=(6, 3.5))
if pooled.size:
    ax.hist(pooled, bins=30, color="#4d4d4d", alpha=0.85)
ax.axvline(spacing_n, color="#d6604d", ls="--", label="one grid spacing")
ax.set_xlabel("inter-field distance (normalised)")
ax.set_title(f"κ_sens {SHOW[1]:g}  τ {SHOW[2]}  pooled field separations")
ax.legend()
fig.tight_layout()
plt.show()

mra = np.asarray(sc["mra"], dtype=float)
radius_m = np.arange(mra.shape[1]) * (ENV_SIZE / sc["bins"])
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(radius_m, np.nanmedian(mra, axis=0), color="#2166ac")
ax.axvline(GRID_SPACING, color="#d6604d", ls="--", label="one grid spacing")
ax.set_xlabel("radius (m)")
ax.set_ylabel("modified radial autocorrelation")
ax.set_title(f"κ_sens {SHOW[1]:g}  τ {SHOW[2]}  median across cells")
ax.legend()
fig.tight_layout()
plt.show()
